# Automatic Alternate Abbreviation Annotation

I am using Claude to try to identify alias symbols of genes that are just variations of abbreviations of the official(primary) gene symbols and the official gene name. The goal is to be able to do this automatically instead of manually. 
These symbols represent the gene with a different symbol than the primary gene symbol. For example, the official long gene name of HGNC:5 is “alpha-1-B glycoprotein” with a primary gene symbol of A1BG. One of the aliases is A1B which is found in PMID: 2447112 to represent “Alpha 1-beta glycoprotein”. This is the same official name of the gene but the abbreviation is different, A1B vs A1BG. 


In [1]:
import re
import requests
import math
from collections.abc import Mapping
from enum import StrEnum
from pathlib import Path
from typing import Any
import time
import ast
import pickle
from dataclasses import dataclass

import polars as pl
from pydantic import BaseModel, ConfigDict
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import BasePromptTemplate, build_empty_registry
from wags_llm.services import StructuredTaskRunner
from rapidfuzz.distance import LCSseq

## Classes (some are in the functions section)

In [2]:
class SkipReason(StrEnum):
    """Reason for why LLM invocation was skipped"""

    HSA_PREFIX = "hsa_prefix"
    EXTRA_CHARACTERS = "extra_characters"
    LOW_LCS_SIMILARITY = "low_lcs_similarity"


class AlternateAbbreviationPredictionResult(BaseModel):
    """Model for LLM and human curator result for determining whether an
    alias symbol corresponds to the primary gene symbol.
    """

    model_config = ConfigDict(extra="forbid", use_enum_values=True)

    llm_annotation: bool | None = None
    llm_skip_reason: SkipReason | None = None
    error_message: str | None = None
    lcs_similarity_score: float | None = None

In [3]:
@dataclass
class RunResult:
    temperature: float
    run_idx: int

    llm_accuracy: float
    llm_coverage: float | None
    llm_summary: any
    llm_metrics: pl.DataFrame

    system_accuracy: float | None = None
    system_summary: any = None
    system_metrics: pl.DataFrame | None = None

## Helper Functions

### Pre-Processing

In [4]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    time.sleep(0.5)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

In [5]:
def lcs_similarity(s1, s2):
    return LCSseq.normalized_similarity(s1, s2)

def has_extra_characters(alias, gene_name):
    norm_alias = re.sub(r'[^A-Z0-9]', '', alias.upper())

    alias_set = set(norm_alias)
    name_set = set(gene_name)

    extra_chars = alias_set - name_set

    return len(extra_chars)

def should_call_llm(alias_symbol, primary_gene_symbol, gene_name, threshold=0.20):
    """Determine whether an alias symbol should be sent to the LLM for further evaluation.

    Applies a series of heuristic filters to the alias symbol
    - checks for excluded prefixes ("HSA-", this is not an alternate abbreviation alias 
    because it is an identifier being used as an alias)
    - extra characters in the alias, not in the gene name
    - longest common subsequence (LCS) similarity to the primary gene symbol to be at least 20%

    :param alias_symbol: The alias gene symbol being evaluated
    :param primary_gene_symbol: The official primary gene symbol
    :param gene_name: The full gene name associated with the gene
    :param threshold: Minimum LCS similarity score required to pass filtering
    :return: A tuple containing:
        - bool: Whether the alias should be sent to the LLM
        - float: The computed LCS similarity score
        - str: The reason code for the decision
    """

    alias = alias_symbol.upper()
    primary = primary_gene_symbol.upper()
    name = gene_name.upper()

    if alias.startswith("HSA-"):
        return False, 0, SkipReason.HSA_PREFIX

    extra_count = has_extra_characters(alias_symbol, name)

    # block only if ≥2 extra characters
    if extra_count >= 2:
        return False, 0, SkipReason.EXTRA_CHARACTERS

    score = lcs_similarity(alias, primary)

    if score < threshold:
        return False, score, SkipReason.LOW_LCS_SIMILARITY

    return True, score, None

### LLM

In [6]:
PROMPT_NAME = "alias_symbol_annotation:alternate_abbreviation"
PROMPT_VERSION = "v1"


class AlternateAbbreviationPromptV1(BasePromptTemplate):
    """Version 1 prompt for predicting alternate abbreviation relationships."""

    version = PROMPT_VERSION
    name = PROMPT_NAME

    def build_system_prompt(self) -> str:
        """Build the system prompt for predicting whether an alias is an alternate abbreviation of the primary gene symbol.

        :returns: System prompt text.
        """
        return (
            "Role: You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias symbol resolution.\n"
            "You will get fired if your accuracy is less than 95%.\n"
            "Background:\n"
            "An alternate abbreviation is when an alias symbol represents the same official gene name or the\n"
            "primary gene symbol but with different letters. If the alias symbol is representing a different description or a previous name of\n"
            "the gene then it is not an alternate abbreviation. Be careful with alias symbols that seem to be shortened versions\n"
            "of the primary gene symbol, they may be a family name and therefore not an alternate abbreviation. Alias symbols\n"
            "have extra characters may be alternate abbreviations unless they have characters that are not present in the official\n"
            "gene name provided in the prompt. Keep in mind, a gene name with a number after the name is not the same gene as a gene name\n"
            "with no numbers or different numbers after the name.\n"
            "Task:\n"
            "Determine whether the alias symbol is an abbreviation variant of the official HGNC gene name.\n"
        )

    def build_user_prompt(
        self,
        payload: Mapping[str, Any],
    ) -> str:
        """Build the user prompt for a single alias symbol.

        :param payload: The alias symbol, HGNC ID, primary gene symbol, official gene name to be evaluated, 
        :returns: User prompt text
        """
        return (
            f"Alias Symbol: {payload['gene_symbol']}\n"
            f"Primary Gene Symbol: {payload['primary_gene_symbol']}\n"
            f"Official Gene Name: {payload['gene_name']}\n"
            f"HGNC ID: {payload['hgnc_id']}\n"
        )

In [7]:
def get_alt_abbreviation_annotation(
    task_runner: StructuredTaskRunner,
    gene_symbol: str,
    primary_gene_symbol: str,
    gene_name: str,
    hgnc_id: str,
) -> AlternateAbbreviationPredictionResult:
    """Annotate a single alias relationship."""

    should_call, score, skip_reason = should_call_llm(
        gene_symbol,
        primary_gene_symbol,
        gene_name,
    )

    if not should_call:
        return AlternateAbbreviationPredictionResult(
            llm_skip_reason=skip_reason,
            lcs_similarity_score=score,
        )

    try:
        task_result = task_runner.execute(
            prompt_name=PROMPT_NAME,
            prompt_version=PROMPT_VERSION,
            payload={
                "gene_symbol": gene_symbol,
                "primary_gene_symbol": primary_gene_symbol,
                "gene_name": gene_name,
                "hgnc_id": hgnc_id,
            },
            response_model=AlternateAbbreviationPredictionResult,
        )

    except Exception as e:
        return AlternateAbbreviationPredictionResult(
            error_message=str(e),
            lcs_similarity_score=score,
        )

    annotation = AlternateAbbreviationPredictionResult.model_validate(
        task_result
    )

    return AlternateAbbreviationPredictionResult(
        llm_annotation=annotation.llm_annotation,
        lcs_similarity_score=score,
    )

In [8]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150
TEMPERATURE = 0.0


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM alternate abbreviation alias annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(AlternateAbbreviationPromptV1())
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )


task_runner = build_llm_task_runner(
    MODEL_ID, REGION_NAME, PROFILE_NAME, MAX_TOKENS, TEMPERATURE
)

In [9]:
def run_single_pass(df, task_runner):
    results = []

    for row in tqdm(df.iter_rows(named=True), total=df.height):
        result = get_alt_abbreviation_annotation(
            task_runner=task_runner,
            gene_symbol=row["gene_symbol"],
            primary_gene_symbol=row["primary_gene_symbol"],
            gene_name=row["gene_name"],
            hgnc_id=row["HGNC_ID"],
        )
        results.append(result)

    return results

In [10]:
def run_experiments(df, temperatures, num_runs):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for row in tqdm(
                df.iter_rows(named=True),
                total=df.height,
                desc=f"T={temp}, run={run_idx+1}",
                leave=False,
            ):

                result = get_alt_abbreviation_annotation(
                    task_runner=task_runner,
                    gene_symbol=row["gene_symbol"],
                    primary_gene_symbol=row["primary_gene_symbol"],
                    gene_name=row["gene_name"],
                    hgnc_id=row["HGNC_ID"],
                )

                results.append(result)

            stored_runs.append({
                "run_idx": run_idx,
                "temperature": temp,
                "results": results,
            })

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

### Result Analysis

In [11]:
def create_analysis_summary(cm: pl.DataFrame) -> pl.DataFrame:
    """Compute per-class recall-style summary from a confusion matrix.

    :param cm: Confusion matrix (square DataFrame)
    :return: Summary DataFrame per class
    """
    rows = []

    classes = cm.columns[1:]

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height == 0:
            numerator = 0
            denominator = 0
        else:
            numerator = row.select(pl.col(cls)).item()

            denominator = row.select(pl.sum_horizontal(classes)).item()

        rows.append(
            {
                "consensus_w_curator": cls,
                "numerator": numerator,
                "denominator": denominator,
                "percentage": (
                    (numerator or 0) / denominator * 100
                    if denominator is not None and denominator > 0
                    else 0.0
                ),
            }
        )

    return pl.DataFrame(rows)

In [12]:
def compute_overall_accuracy(cm: pl.DataFrame) -> float:
    """Compute overall accuracy from a confusion matrix.

    :param cm: Confusion matrix DataFrame (square matrix)
    :return: Accuracy as a float between 0 and 1
    """
    classes = cm.columns[1:]

    total = cm.select(pl.sum_horizontal(classes)).to_series().sum()

    correct = 0

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height > 0:
            correct += row.select(pl.col(cls)).item() or 0

    return correct / total if total > 0 else math.nan

In [13]:
def boolean_confusion_matrix(df: pl.DataFrame, gt_col: str, pred_col: str) -> pl.DataFrame:
    """
    Compute a boolean confusion matrix in the traditional matrix style.

    Parameters:
    - df: pandas DataFrame
    - gt_col: str, name of the ground truth column
    - pred_col: str, name of the predicted column

    Returns:
    - confusion_matrix: pandas DataFrame with row/column labels
    """

    df_clean = (
        df.filter(
            pl.col(gt_col).is_not_null() &
            pl.col(pred_col).is_not_null()
        )
        .with_columns([
            pl.col(gt_col).cast(pl.Boolean).alias("gt"),
            pl.col(pred_col).cast(pl.Boolean).alias("pred"),
        ])
    )

    return (
        df_clean
        .group_by(["gt", "pred"])
        .len()
        .pivot(
            values="len",
            index="gt",
            on="pred",
            aggregate_function="sum",
        )
        .fill_null(0)
    )

In [14]:
def create_system_level_predictions(
    df: pl.DataFrame,
    pred_col: str,
    system_pred_col: str = "llm_system",
) -> pl.DataFrame:
    """Create a system-level prediction column where null predictions
    are treated as False.

    :param df: Input dataframe
    :param pred_col: Original prediction column
    :param system_pred_col: Name of new system-level prediction column
    :return: DataFrame with added system-level prediction column
    """
    return df.with_columns(
        pl.col(pred_col)
        .fill_null(False)
        .alias(system_pred_col)
    )

In [15]:
def compute_coverage(
    df: pl.DataFrame,
    pred_col: str,
) -> float:
    """Compute proportion of rows with non-null predictions."""

    return (
        df
        .select(
            pl.col(pred_col)
            .is_not_null()
            .mean()
        )
        .item()
    )

In [16]:
def analyze_results(
    df: pl.DataFrame,
) -> tuple[
    pl.DataFrame,
    float,
    pl.DataFrame,
    pl.DataFrame,
    float,
    pl.DataFrame,
    float, 
]:
    """Evaluate LLM predictions against ground truth using:
    1. Conditional LLM-only evaluation
    2. System-level evaluation (null -> False)
    """

    analysis_df = df.clone()

    # CONDITIONAL LLM EVALUATION

    llm_cm = boolean_confusion_matrix(
        analysis_df,
        "gt",
        "llm",
    )

    TN = llm_cm.filter(pl.col("gt") == False)["false"].item()
    FP = llm_cm.filter(pl.col("gt") == False)["true"].item()
    FN = llm_cm.filter(pl.col("gt") == True)["false"].item()
    TP = llm_cm.filter(pl.col("gt") == True)["true"].item()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    llm_metrics_df = pl.DataFrame({
        "evaluation_type": ["conditional_llm"],
        "precision": [precision],
        "recall": [recall],
        "f1": [f1],
        "TP": [TP],
        "FP": [FP],
        "TN": [TN],
        "FN": [FN],
    })

    llm_match_analysis_summary = create_analysis_summary(llm_cm)

    llm_accuracy = compute_overall_accuracy(llm_cm)

    llm_coverage = compute_coverage(
    analysis_df,
    "llm",
    )   

    # SYSTEM-LEVEL EVALUATION (null -> False)

    system_df = create_system_level_predictions(
        analysis_df,
        pred_col="llm",
        system_pred_col="llm_system",
    )

    system_cm = boolean_confusion_matrix(
        system_df,
        "gt",
        "llm_system",
    )

    system_TN = system_cm.filter(pl.col("gt") == False)["false"].item()
    system_FP = system_cm.filter(pl.col("gt") == False)["true"].item()
    system_FN = system_cm.filter(pl.col("gt") == True)["false"].item()
    system_TP = system_cm.filter(pl.col("gt") == True)["true"].item()

    system_precision = (
        system_TP / (system_TP + system_FP)
        if (system_TP + system_FP) > 0
        else 0.0
    )

    system_recall = (
        system_TP / (system_TP + system_FN)
        if (system_TP + system_FN) > 0
        else 0.0
    )

    system_f1 = (
        2 * system_precision * system_recall /
        (system_precision + system_recall)
        if (system_precision + system_recall) > 0
        else 0.0
    )

    system_metrics_df = pl.DataFrame({
        "evaluation_type": ["system_level"],
        "precision": [system_precision],
        "recall": [system_recall],
        "f1": [system_f1],
        "TP": [system_TP],
        "FP": [system_FP],
        "TN": [system_TN],
        "FN": [system_FN],
        })

    system_match_analysis_summary = create_analysis_summary(system_cm)

    system_accuracy = compute_overall_accuracy(system_cm)


    return (
        llm_match_analysis_summary,
        llm_accuracy,
        llm_metrics_df,
        llm_coverage, 
        system_match_analysis_summary,
        system_accuracy,
        system_metrics_df,   
    )

In [17]:
def process_results(
    df: pl.DataFrame,
) -> tuple[dict, dict, list]:
    """Run analysis on a provided model output DataFrame.

    :param df: Input DataFrame containing model results
    :param collapse: Whether to collapse categories during analysis
    :return: Tuple of all results (summaries, cm, accuracies)
    """
    llm_match_analysis_summary, llm_accuracy, llm_metrics_df, llm_coverage, system_match_analysis_summary, system_accuracy, system_metrics_df = analyze_results(df)

    return (
        llm_match_analysis_summary,
        llm_accuracy,
        llm_metrics_df,
        llm_coverage, 
        system_match_analysis_summary,
        system_accuracy,
        system_metrics_df,   
    )

In [18]:
def build_evaluation_df(df: pl.DataFrame, stored_runs: list, run_idx: int = -1) -> pl.DataFrame:
    """Attach LLM outputs + diagnostics back onto original dataframe."""

    run = stored_runs[run_idx]

    results = run["results"]

    eval_rows = []

    for row, res in zip(df.iter_rows(named=True), results):

        llm_label = getattr(res, "llm_annotation", None)

        skip_reason = getattr(res, "llm_skip_reason", None)

        eval_rows.append({
            **row,
            "llm": llm_label,
            "llm_skip_reason": skip_reason,
        })

    return pl.DataFrame(eval_rows)

In [19]:
def add_correctness_column(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        (
            pl.col("llm") ==
            pl.col("alternate_abbreviation_status_w_no_context")
        ).alias("llm_correct")
    )

In [20]:
def build_eval_df(
    short_df: pl.DataFrame,
    stored_runs: list,
    temperature: float,
    run_idx: int,
) -> pl.DataFrame:

    run = next(
        (r for r in stored_runs
         if r["temperature"] == temperature and r["run_idx"] == run_idx),
        None,
    )

    if run is None:
        available = [(r["temperature"], r["run_idx"]) for r in stored_runs]
        raise ValueError(
            f"No run found for (temperature={temperature}, run_idx={run_idx}). "
            f"Available runs: {available}"
        )

    results = run["results"]

    eval_rows = []

    for row, res in zip(short_df.iter_rows(named=True), results):
        eval_rows.append({
            **row,
            "llm": getattr(res, "llm_annotation", None),
            "llm_skip_reason": getattr(res, "llm_skip_reason", None),
            "temperature": temperature,
            "run_idx": run_idx,
        })

    eval_df = pl.DataFrame(eval_rows)

    return add_correctness_column(eval_df)

## Add more alias symbols to annotate

Only run this section if you want to add more samples to manually curate

In [21]:
SKIP_ADD_MORE_SAMPLES = True
if not SKIP_ADD_MORE_SAMPLES:
    NUMBER_OF_NEW_SAMPLES_TO_ADD = 0 # Add number of new samples you want to add

In [22]:
ID_COLS = ["HGNC_ID", "ENSG_ID", "NCBI_ID"]

# Load and clean capture df
## This is the file with alias and primary gene symbol pairs and what categories the aliases were captured as
## Generated in the 5_symbol_capture_analysis.ipynb
## Clean it up with converting to booleans and renaming columns for clarity
if not SKIP_ADD_MORE_SAMPLES:
    capture_df = (
        pl.read_csv("../output/summary_df.csv")
        .rename({
            "captured": "captured_status",
            "captured as:": "captured_category_list",
        })
        .drop("")
        .with_columns(
            pl.when(pl.col("captured_status") == "T")
            .then(True)
            .when(pl.col("captured_status") == "F")
            .then(False)
            .otherwise(None)
            .alias("captured_status"),

            pl.col(ID_COLS).map_elements(
                lambda x: (
                    ", ".join(sorted(ast.literal_eval(x)))
                    if x is not None
                    else None
                ),
                return_dtype=pl.String,
            ),
        )
    )

    # Sample new aliases
    ## To add to the sample set for manual annotation (68- wanted to get to a total of 500)

    new_samples_df = (
        capture_df
        .filter(
            (pl.col("captured_category_list") != "Primary Gene Symbol")
            | pl.col("captured_category_list").is_null()
        )
        .group_by("captured_status")
        .map_groups(
            lambda g: g.sample(
                n=min(len(g), (NUMBER_OF_NEW_SAMPLES_TO_ADD/2)),
                seed=41,
            )
        )
    )

    # Load curated annotations

    curated_df = pl.read_excel(
        "/Users/rsaxs014/Desktop/gene-harmony-analysis/output/alt_abbrev_annotation_manually_annotated_df.xlsx"
    )

    # Add gene_name column safely to new samples df

    if "gene_name" not in new_samples_df.columns:
        new_samples_df = new_samples_df.with_columns(pl.lit(None).alias("gene_name"))

    # Make sure the newly added samples don't already exist in the manually curated set

    new_samples_df = (
        new_samples_df
        .join(
            curated_df,
            on=["HGNC_ID", "captured_status"],
            how="anti",
        )
        .with_columns(
            pl.lit(None).alias(
                "alternate_abbreviation_status_w_no_context"
            )
        )
        .select(curated_df.columns)
    )

    # Combine dfs

    df = pl.concat(
        [curated_df, new_samples_df],
        how="vertical",
    )

    # Add gene names, by HGNC ID, to the annotation set for easier manual review

    missing_ids = (
        df
        .filter(pl.col("gene_name").is_null())
        .select("HGNC_ID")
        .unique()
        .to_series()
        .to_list()
    )

    gene_map = {
        hgnc_id: get_gene_name(hgnc_id)
        for hgnc_id in tqdm(missing_ids)
    }

    df = df.with_columns(
        pl.col("gene_name")
        .fill_null(pl.col("HGNC_ID").replace(gene_map))
    )

    # Export

    df.write_excel(
        "../output/alt_abbrev_annotation_to_annotate_df.xlsx"
    )
    f"The original set had {curated_df.height} samples, the new set has {df.height} samples after adding new ones and removing any that were already in the curated set({new_samples_df.height} new samples)"

**After manually annotating the new rows in alt_abbrev_annotation_to_annotate_df.xlsx, it is renamed to alt_abbrev_annotation_manually_annotated_df.xlsx**

## Run LLM with gene symbols, name, and prompt

In [23]:
# # For testing
# df = pl.read_excel(
#     "../output/alt_abbrev_annotation_manually_annotated_df.xlsx"
# )
# short_df = df.head(30)
# short_df.write_excel(
#     "../output/short_alt_abbrev_annotation_manually_annotated_df.xlsx"
# )

In [24]:
# # Using the manually curated set of samples
# SHORT_SAMPLE_PATH = Path(
#     "/Users/rsaxs014/Desktop/gene-harmony-analysis/output/short_alt_abbrev_annotation_manually_annotated_df.xlsx"
# )

In [25]:
# Using the manually curated set of samples
SAMPLE_PATH = Path(
    "/Users/rsaxs014/Desktop/gene-harmony-analysis/output/alt_abbrev_annotation_manually_annotated_df.xlsx"
)


In [26]:
df = pl.read_excel(SAMPLE_PATH)

In [27]:
LLM_RUN_RESULTS_DIR = Path("llm_run_results")
LLM_RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [28]:
TEMPERATURES = [0.0]
NUM_RUNS = 3

temp_str = "-".join(str(t).replace(".", "p") for t in TEMPERATURES)

stored_runs_path = Path(
    f"stored_runs_t{temp_str}_n{NUM_RUNS}.pkl"
)

if stored_runs_path.exists():
    print(f"Loading {stored_runs_path}")

    with open(stored_runs_path, "rb") as f:
        stored_runs = pickle.load(f)

else:
    print("Running experiments...")

    stored_runs = run_experiments(
        df,
        TEMPERATURES,
        NUM_RUNS,
    )

    with open(stored_runs_path, "wb") as f:
        pickle.dump(stored_runs, f)

Loading stored_runs_t0p0_n3.pkl


In [29]:
all_runs = []

for path in Path(".").glob("stored_runs_*.pkl"):
    with open(path, "rb") as f:
        all_runs.extend(pickle.load(f))

unique_runs = {
    (
        run["temperature"],
        run["run_idx"],
    )
    for run in all_runs
}

print(f"Total number of runs stored: {len(all_runs)}")
print(f"Unique runs: {unique_runs}")

Total number of runs stored: 3
Unique runs: {(0.0, 1), (0.0, 2), (0.0, 0)}


In [30]:
rows = []

for run in stored_runs:
    for i, r in enumerate(run["results"]):
        rows.append({
            "row": i,
            "run": run["run_idx"],
            "temp": run["temperature"],
            "llm": r.llm_annotation,
            "gt": df[i, "alternate_abbreviation_status_w_no_context"],
        })

df_runs = pl.DataFrame(rows).with_columns([
    pl.col("llm").cast(pl.Boolean),
])

In [31]:
results_by_run = {}

for temp, run_idx in unique_runs:

    run_df = df_runs.filter(
        (pl.col("temp") == temp) &
        (pl.col("run") == run_idx)
    )

    (
        llm_summary,
        llm_acc,
        llm_metrics,
        llm_cov,
        system_summary,
        system_acc,
        system_metrics,
    ) = analyze_results(run_df)

    run_result = RunResult(
        temperature=temp,
        run_idx=run_idx,
        llm_accuracy=llm_acc,
        llm_coverage=llm_cov,
        llm_summary=llm_summary,
        llm_metrics=llm_metrics,
        system_accuracy=system_acc,
        system_summary=system_summary,
        system_metrics=system_metrics,
    )

    results_by_run[(temp, run_idx)] = run_result

In [32]:
all_results = []

for run in results_by_run.values():
    all_results.append({
        "temperature": run.temperature,
        "run_idx": run.run_idx,
        "llm_accuracy": run.llm_accuracy,
        "llm_coverage": run.llm_coverage,
        "system_accuracy": run.system_accuracy,
    })

results_df = pl.DataFrame(all_results)
print(results_df)

shape: (3, 5)
┌─────────────┬─────────┬──────────────┬──────────────┬─────────────────┐
│ temperature ┆ run_idx ┆ llm_accuracy ┆ llm_coverage ┆ system_accuracy │
│ ---         ┆ ---     ┆ ---          ┆ ---          ┆ ---             │
│ f64         ┆ i64     ┆ f64          ┆ f64          ┆ f64             │
╞═════════════╪═════════╪══════════════╪══════════════╪═════════════════╡
│ 0.0         ┆ 1       ┆ 0.923497     ┆ 0.363095     ┆ 0.972222        │
│ 0.0         ┆ 2       ┆ 0.918033     ┆ 0.363095     ┆ 0.970238        │
│ 0.0         ┆ 0       ┆ 0.918033     ┆ 0.363095     ┆ 0.970238        │
└─────────────┴─────────┴──────────────┴──────────────┴─────────────────┘


In [33]:
run_00_2 = results_by_run[(0.0, 2)]
print(f"LLM Coverage: {run_00_2.llm_coverage}")
print(run_00_2.llm_metrics)
print(run_00_2.llm_accuracy)
print(run_00_2.system_metrics)
print(run_00_2.system_accuracy)

LLM Coverage: 0.3630952380952381
shape: (1, 8)
┌─────────────────┬───────────┬──────────┬──────────┬─────┬─────┬─────┬─────┐
│ evaluation_type ┆ precision ┆ recall   ┆ f1       ┆ TP  ┆ FP  ┆ TN  ┆ FN  │
│ ---             ┆ ---       ┆ ---      ┆ ---      ┆ --- ┆ --- ┆ --- ┆ --- │
│ str             ┆ f64       ┆ f64      ┆ f64      ┆ i64 ┆ i64 ┆ i64 ┆ i64 │
╞═════════════════╪═══════════╪══════════╪══════════╪═════╪═════╪═════╪═════╡
│ conditional_llm ┆ 0.90625   ┆ 0.865672 ┆ 0.885496 ┆ 58  ┆ 6   ┆ 110 ┆ 9   │
└─────────────────┴───────────┴──────────┴──────────┴─────┴─────┴─────┴─────┘
0.9180327868852459
shape: (1, 8)
┌─────────────────┬───────────┬──────────┬──────────┬─────┬─────┬─────┬─────┐
│ evaluation_type ┆ precision ┆ recall   ┆ f1       ┆ TP  ┆ FP  ┆ TN  ┆ FN  │
│ ---             ┆ ---       ┆ ---      ┆ ---      ┆ --- ┆ --- ┆ --- ┆ --- │
│ str             ┆ f64       ┆ f64      ┆ f64      ┆ i64 ┆ i64 ┆ i64 ┆ i64 │
╞═════════════════╪═══════════╪══════════╪══════════╪═════╪═══

In [34]:
eval_df = build_eval_df(
    df,
    stored_runs,
    temperature=0.0,
    run_idx=1,
)
eval_df

HGNC_ID,ENSG_ID,NCBI_ID,captured_status,captured_category_list,primary_gene_symbol,gene_symbol,alternate_abbreviation_status_w_no_context,gene_name,llm,llm_skip_reason,temperature,run_idx,llm_correct
str,str,str,bool,str,str,str,bool,str,bool,str,f64,i64,bool
"""HGNC:12406""","""ENSG00000166402""","""GENE ID:7275""",false,null,"""TUB""","""rd5""",false,"""TUB bipartite transcription fa…",null,"""extra_characters""",0.0,1,null
"""HGNC:6631""","""ENSG00000074695""","""GENE ID:3998""",false,null,"""LMAN1""","""GP58""",false,"""lectin, mannose binding 1""",null,"""extra_characters""",0.0,1,null
"""HGNC:9911""","""ENSG00000216835""","""GENE ID:3186""",false,null,"""RBMXP1""","""HNRNP-G""",false,"""RBMX pseudogene 1""",false,null,0.0,1,true
"""HGNC:3795""","""ENSG00000110203""","""GENE ID:2352""",false,null,"""FOLR3""","""FRgamma""",true,"""folate receptor gamma""",true,null,0.0,1,true
"""HGNC:13919""","""ENSG00000234651, ENSG000000961…","""GENE ID:7917""",false,null,"""BAG6""","""SCYTHE""",false,"""BAG cochaperone 6""",null,"""extra_characters""",0.0,1,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HGNC:22635""","""ENSG00000231167""","""GENE ID:646531""",false,null,"""YBX1P2""","""TCAG_1818537""",false,"""Y-box binding protein 1 pseudo…",null,"""extra_characters""",0.0,1,null
"""HGNC:29334""","""ENSG00000159733""","""GENE ID:57732""",false,null,"""ZFYVE28""","""LYST2""",false,"""zinc finger FYVE-type containi…",null,"""extra_characters""",0.0,1,null
"""HGNC:43475""","""ENSG00000284092""","""GENE ID:100847081""",true,"""Prefix Gene Group Symbol""","""MIR5703""","""mir-5703""",true,"""microRNA 5703""",true,null,0.0,1,true
